#### Pydantic Basics: Creating and using Models

Pydantic models are the foundation of data validation in Python. They use Python type annotations to define the structure and validate data at runtime. Here is a detailed exploration of basic model creation with several examples

In [1]:
from pydantic import BaseModel

In [2]:
class Person(BaseModel):
    name: str
    age: int
    city: str

person = Person(name="Krish", age=35, city="Bangalore")
print(person)

name='Krish' age=35 city='Bangalore'


In [4]:
# Let's create the same class without inheriting the BaseModel class
from dataclasses import dataclass

@dataclass
class Person1:
    name: str
    age: int
    city: str

person1 = Person1(name="Krish", age=35, city="Bangalore")
print(person1)

Person1(name='Krish', age=35, city='Bangalore')


In [ ]:
# This will execute successfully because there is no data type checking here.

person2 = Person1(name="Krish", age=35, city=19)
print(person2)

Person1(name='Krish', age=35, city=19)


In [6]:
# But this will throw error because Pydantic has inbuilt data type checking
person3 = Person(name="Krish", age=35, city=19)
print(person3)

ValidationError: 1 validation error for Person
city
  Input should be a valid string [type=string_type, input_value=19, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

#### Model with Optional fields

Add optional fields using Python's Optional Type:

In [8]:
from typing import Optional

class Employee(BaseModel):
    id: int
    name: str
    department: str
    salary: Optional[float] = None # Optional with default value
    is_active: Optional[bool] = True # Optional with default value "True"

In [9]:
# Examples with and without optional fields
emp1 = Employee(id=1, name="John", department="IT")
print(emp1)

id=1 name='John' department='IT' salary=None is_active=True


In [10]:
emp2 = Employee(id=2, name="Jane", department="HR", salary=60000.0, is_active=False)
print(emp2)

id=2 name='Jane' department='HR' salary=60000.0 is_active=False


Definition:

* Optional[type]: Indicates the field can be None
* Default value (=None or = True): Makes the field optional
* Required fields must still be provided
* Pydantic validates types even for optional fields when values are provided

In [11]:
from pydantic import BaseModel
from typing import List

class Classroom(BaseModel):
    room_number: str
    students: List[str]
    capacity: int

In [13]:
# Create a classroom
classroom = Classroom(
    room_number="A101",
    students=["Alice", "Bob", "Charlie"],
    capacity=30
)

print(classroom)

room_number='A101' students=['Alice', 'Bob', 'Charlie'] capacity=30


In [14]:
# Create a classroom
classroom1 = Classroom(
    room_number="A101",
    students=("Alice", "Bob", "Charlie"), # We provide a tuple to check whether type-casting will happen or not
    capacity=30
)

print(classroom1)

room_number='A101' students=['Alice', 'Bob', 'Charlie'] capacity=30


In [15]:
try:
    invalid_val=Classroom(room_number="A1", students=["Harshit", 12345], capacity=30)
except ValueError as e:
    print(e)

1 validation error for Classroom
students.1
  Input should be a valid string [type=string_type, input_value=12345, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


#### 4. Model with Nested Models

Create complex structures with nested models

In [16]:
from pydantic import BaseModel

class Address(BaseModel):
    street: str
    city: str
    zip_code: str

class Customer(BaseModel):
    customer_id: int
    name: str
    address: Address # Nested model

customer = Customer(
    customer_id=1,
    name="Emma",
    address={"street": "123 main St", "city": "Boston", "zip_code": "02108"}
)

print(customer)

customer_id=1 name='Emma' address=Address(street='123 main St', city='Boston', zip_code='02108')


#### Pydantic Fields: Customization and Constraints

The Field function in Pydantic enhances model fields beyond basic type hints by allowing you to specify validation rules, default values, aliases, and more. Here is a comprehensive tutorial with examples

In [17]:
from pydantic import BaseModel,Field

class Item(BaseModel):
    name: str = Field(min_length=2, max_length=50)
    price:float = Field(gt=0, le=1000)
    quantity: int = Field(ge=0)

item = Item(name="Book", price=50, quantity=10)
print(item)

name='Book' price=50.0 quantity=10


In [18]:
item1 = Item(name="Book", price=-10, quantity=10)
print(item1)

ValidationError: 1 validation error for Item
price
  Input should be greater than 0 [type=greater_than, input_value=-10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than

In [19]:
from pydantic import BaseModel, Field

class User(BaseModel):
    username: str = Field(..., description="Unique username for the user")
    age: int = Field(default=18, description="User age, defaults to 18")
    email: str = Field(default_factory=lambda: "user@example.com", description="Default email address")

user1 = User(username="alice")
print(user1)

user2 = User(username="bob", age=25, email="bob@domain.com")
print(user2)

username='alice' age=18 email='user@example.com'
username='bob' age=25 email='bob@domain.com'


In [20]:
print(User.model_json_schema())

{'properties': {'username': {'description': 'Unique username for the user', 'title': 'Username', 'type': 'string'}, 'age': {'default': 18, 'description': 'User age, defaults to 18', 'title': 'Age', 'type': 'integer'}, 'email': {'description': 'Default email address', 'title': 'Email', 'type': 'string'}}, 'required': ['username'], 'title': 'User', 'type': 'object'}
